# Spark 3.5.x → 4.0 · 4.1 · 4.2: what's new in each version

This notebook tests the main Spark **4.x** changes against the lakehouse stack (Iceberg + Hive Metastore + MinIO + Kafka). Each section heading has a tag showing the release that brought the feature: `[4.0]`, `[4.1]` or `[4.2]`.

> **Cluster used for the outputs below:** Spark **4.1.0**, Java **21** (Temurin), classic mode (`spark://spark-master:7077`). Cells for **4.2-only** features have no output yet. Run them on a 4.2 cluster.

## 3.5.x vs 4.0 vs 4.1 vs 4.2 at a glance

| Area | 3.5.x | 4.0 | 4.1 | 4.2 |
|---|---|---|---|---|
| **Release type** | Maintenance (fixes only) | Major release, has breaking changes | Feature release | Feature release (latest) |
| **Runtime** | Java 8/11/17 · Scala 2.12/2.13 | **Java 17+** · **Scala 2.13 only** · Python 3.9+ | Python 3.10+ | Java 17/21/25 · Arrow-first Python |
| **SQL defaults** | ANSI **off** (`CAST('abc' AS INT)` → `NULL`) | **ANSI on** (bad cast or `10/0` → error) | SQL scripting **on by default** | – |
| **Semi-structured data** | JSON kept as `STRING` | **`VARIANT`** type, `parse_json`, `variant_get` | VARIANT GA, shredding | – |
| **SQL language** | Nested `SELECT … FROM (SELECT …)` | **Pipe `\|>`**, **session variables** (`DECLARE` / `SET VAR`) | **`WITH RECURSIVE`**, **SQL scripting** (`BEGIN … END`, `FOR`, `EXIT HANDLER`), **sketches** (`approx_top_k`, KLL, Theta) | **`QUALIFY`**, pipe aggregates, `time_bucket`, `max_by(x, y, k)`, `IGNORE NULLS`, **vector functions**, **geospatial**, **CDC** (`.changes()`) |
| **Python UDF / UDTF** | Pickled `@udf`, `@pandas_udf`, fixed-schema UDTF | **Polymorphic UDTF** (`analyze()`), **UDF profiler** | **`@arrow_udf`** (works on `pyarrow.Array`) | **Arrow on by default** for plain `@udf` |
| **Data sources** | Custom sources need Scala/Java | **Python Data Source API** (readers/writers in pure Python) | – | – |
| **Streaming** | `(flat)MapGroupsWithState`, continuous mode | **State Data Source** (`state-metadata`, `statestore`, `readChangeFeed`) | **`transformWithState` in Python**, **Real-Time Mode** (Scala/JVM trigger) | `trigger(realTime=…)` in **PySpark** |
| **Logging** | Plain text log lines | **Structured JSON logging** (`spark.log.structuredLogging.enabled`), `SPARK_LOG_SCHEMA` | – | – |
| **Spark Connect** | Basic | Near-full parity, `spark.api.mode`, slim `pyspark-client` | ML on Connect GA, Connect JDBC driver | `toJSON`, `zipWithIndex`, `GetStatus` API |

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    IntegerType,
    DoubleType,
    TimestampType,
)
from pyspark.sql import functions as F

In [2]:
import os
CATALOG = "lakehouse"
C = f"spark.sql.catalog.{CATALOG}"

spark = (
    SparkSession.builder
    .appName("Lakehouse Implementation")
    .master("spark://spark-master:7077")
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")

    # Named Iceberg catalog backed by the Hive Metastore
    .config(C, "org.apache.iceberg.spark.SparkCatalog")
    .config(f"{C}.type", "hive")
    .config(f"{C}.uri", "thrift://hive-metastore:9083")
    .config(f"{C}.warehouse", "s3://buck1/warehouse")

    # Iceberg's own S3 client for table data and metadata.
    # Keys and region come from AWS_ACCESS_KEY_ID / AWS_SECRET_ACCESS_KEY / AWS_REGION.
    .config(f"{C}.io-impl", "org.apache.iceberg.aws.s3.S3FileIO")
    .config(f"{C}.s3.endpoint", os.environ["S3_ENDPOINT"])
    .config(f"{C}.s3.path-style-access", "true")
    .config(f"{C}.http-client.type", "apache")   # the only HTTP client in bundle-2.29.52.jar

    .config("spark.sql.defaultCatalog", CATALOG)

    # S3A is still used for streaming checkpoints (Spark writes them through Hadoop, not Iceberg).
    # S3A's default credential chain reads the same AWS_* environment variables.
    .config("spark.hadoop.fs.s3a.endpoint", os.environ["S3_ENDPOINT"])
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false")

    .getOrCreate()
)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/10/22 05:40:48 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
print("Spark version:", spark.version)

Spark version: 4.1.0


## Java Runtime Modernization from java 11 -> java 21

In [4]:
import os
import subprocess

print("Spark version:", spark.version)

# Java selected by your current shell environment
print(subprocess.run(
    ["java", "-version"],
    capture_output=True,
    text=True
).stderr)

print("JAVA_HOME:", os.environ.get("JAVA_HOME"))

# Java version used by the running Spark driver JVM
print(
    "Driver JVM version:",
    spark._jvm.java.lang.System.getProperty("java.version")
)

print(
    "Java vendor:",
    spark._jvm.java.lang.System.getProperty("java.vendor")
)

print(
    "JVM name:",
    spark._jvm.java.lang.System.getProperty("java.vm.name")
)

Spark version: 4.1.0
openjdk version "21.0.9" 2025-10-21 LTS
OpenJDK Runtime Environment Temurin-21.0.9+10 (build 21.0.9+10-LTS)
OpenJDK 64-Bit Server VM Temurin-21.0.9+10 (build 21.0.9+10-LTS, mixed mode, sharing)

JAVA_HOME: /opt/java/openjdk
Driver JVM version: 21.0.9
Java vendor: Eclipse Adoptium
JVM name: OpenJDK 64-Bit Server VM


## ANSI mode by default True [4.0]

In [19]:
spark.conf.get("spark.sql.ansi.enabled")
spark.sql("SELECT CAST('abc' AS INT);").show() # will throw an error
spark.sql("SELECT 10 / 0;").show() # will throw an error

'true'

In [16]:
spark.conf.set("spark.sql.ansi.enabled", "false")
spark.sql("SELECT CAST('abc' AS INT);").show()
spark.sql("SELECT 10 / 0;").show()

+----------------+
|CAST(abc AS INT)|
+----------------+
|            NULL|
+----------------+

+--------+
|(10 / 0)|
+--------+
|    NULL|
+--------+



## Variant Data Type [4.0,4.1]

In [4]:
from pyspark.sql import functions as F

json_data = [
    ('{"user_id": 101, "name": "Ali", "city": "Lahore"}',),
    ('{"user_id": 102, "name": "Sara", "device": "iPhone"}',),
    ('{"user_id": 103, "name": "Ahmed", "premium": true}',),
    ('{"user_id": 104, "name": "Usman", "age": 25}',)
]

df = spark.createDataFrame(json_data, ["json_string"])

In [8]:
# Convert JSON strings into Spark VARIANT values
variant_df = df.select(
    F.parse_json("json_string").alias("payload")
)

variant_df.printSchema()
variant_df.show(truncate=False)

root
 |-- payload: variant (nullable = true)



+-----------------------------------------------+
|payload                                        |
+-----------------------------------------------+
|{"city":"Lahore","name":"Ali","user_id":101}   |
|{"device":"iPhone","name":"Sara","user_id":102}|
|{"name":"Ahmed","premium":true,"user_id":103}  |
|{"age":25,"name":"Usman","user_id":104}        |
+-----------------------------------------------+



In [10]:
variant_df.createOrReplaceTempView("variant_events")

In [11]:
result = spark.sql("""
    SELECT
        variant_get(payload, '$.user_id', 'INT') AS user_id,
        variant_get(payload, '$.name', 'STRING') AS name,
        variant_get(payload, '$.city', 'STRING') AS city,
        variant_get(payload, '$.device', 'STRING') AS device,
        variant_get(payload, '$.premium', 'BOOLEAN') AS premium,
        variant_get(payload, '$.age', 'INT') AS age
    FROM variant_events
""")

result.show(truncate=False)

+-------+-----+------+------+-------+----+
|user_id|name |city  |device|premium|age |
+-------+-----+------+------+-------+----+
|101    |Ali  |Lahore|NULL  |NULL   |NULL|
|102    |Sara |NULL  |iPhone|NULL   |NULL|
|103    |Ahmed|NULL  |NULL  |true   |NULL|
|104    |Usman|NULL  |NULL  |NULL   |25  |
+-------+-----+------+------+-------+----+



## pipe `|>` and session variables [4.0]

* **Pipe syntax**: write a query top-to-bottom in the order it runs (`FROM → WHERE → AGGREGATE → ORDER`) instead of the inside-out `SELECT … FROM (SELECT …)`.
* **Session variables**: `DECLARE VARIABLE` / `SET VAR` hold values inside SQL. On 3.5 you had to build the SQL string in Python.
* *No speed difference*: both compile to the same plans as the old forms.

In [7]:
data = [
    (1, 150.0, '{"city": "Lahore"}'),
    (2, 300.0, '{"city": "Karachi"}'),
    (3, 450.0, '{"city": "Lahore"}'),
    (4, 200.0, '{"city": "Islamabad"}'),
    (5, 600.0, '{"city": "Karachi"}'),
    (6, 250.0, '{"city": "Lahore"}'),
    (7, 100.0, '{"city": "Islamabad"}'),
]

columns = ["order_id", "amount", "payload"]

df = spark.createDataFrame(data, columns)

df.createOrReplaceTempView("orders")

# 3. Declare and update the session variable
spark.sql("""
    DECLARE OR REPLACE VARIABLE min_amount DOUBLE DEFAULT 100
""")

spark.sql("SET VAR min_amount = 250")

# 4. Run your query
spark.sql("""
    FROM orders
    |> WHERE amount > min_amount
    |> EXTEND get_json_object(payload, '$.city') AS city
    |> AGGREGATE COUNT(*) AS orders,
                  ROUND(SUM(amount), 2) AS revenue
       GROUP BY city
    |> ORDER BY revenue DESC
""").show()

+-------+------+-------+
|   city|orders|revenue|
+-------+------+-------+
|Karachi|     2|  900.0|
| Lahore|     1|  450.0|
+-------+------+-------+



## 4.1: recursive CTEs, SQL scripting (GA, on by default), sketch functions

* **`WITH RECURSIVE`**: hierarchies (org charts, bill of materials, graph walks) in plain SQL. Limits: `spark.sql.cteRecursionLevelLimit` (100), `spark.sql.cteRecursionRowLimit` (1M).
* **SQL scripting**: `BEGIN … END` blocks with variables, `IF`, `WHILE`, `FOR`, `LOOP`, and **`EXIT` handlers**, sent as one `spark.sql(...)` call.
* **Sketches**: `approx_top_k`, KLL quantiles (`kll_sketch_*`), Theta distinct counts (`theta_*`). Sketches can be **stored and merged later**, e.g. daily sketches → monthly answer.

In [8]:
# Recursive CTE: walk a category tree of arbitrary depth
data = [
    (1, "All", None),
    (2, "Phones", 1),
    (3, "Android", 2),
    (4, "iOS", 2),
    (5, "Laptops", 1),
    (6, "Gaming", 5)
]

cats = spark.createDataFrame(data, ["id", "name", "parent"])
cats.createOrReplaceTempView("cats")

spark.sql("""
WITH RECURSIVE tree(id, name, parent, depth) AS (
    -- Start with the root category
    SELECT id, name, parent, 0
    FROM cats
    WHERE parent IS NULL

    UNION ALL

    -- Find each category's children
    SELECT c.id, c.name, c.parent, t.depth + 1
    FROM cats c
    JOIN tree t ON c.parent = t.id
)
SELECT name, depth
FROM tree
ORDER BY depth, name
""").show()

+-------+-----+
|   name|depth|
+-------+-----+
|    All|    0|
|Laptops|    1|
| Phones|    1|
|Android|    2|
| Gaming|    2|
|    iOS|    2|
+-------+-----+



In [ ]:
# SQL scripting: loop + EXIT handler, as a single statement
script = """
BEGIN
    DECLARE total DOUBLE DEFAULT 0;
    DECLARE status STRING DEFAULT 'ok';

    FOR c AS (
        SELECT DISTINCT get_json_object(payload, '$.city') AS city
        FROM ev_var
    ) DO
        SET total = total + (
            SELECT SUM(amount)
            FROM ev_var
            WHERE get_json_object(payload, '$.city') = c.city
        );
    END FOR;

    BEGIN
        DECLARE EXIT HANDLER FOR DIVIDE_BY_ZERO
            SET status = 'error caught';

        SELECT 1 / 0;
    END;

    SELECT total AS revenue, status;
END
"""

spark.sql(script).show(truncate=False)

In [9]:
# Sketches: top-k cities, median amount, distinct customers, mergeable
data = [
    (1, "Lahore", 100.0, "C1"),
    (2, "Karachi", 200.0, "C2"),
    (3, "Lahore", 300.0, "C1"),
    (4, "Lahore", 400.0, "C3"),
    (5, "Karachi", 500.0, "C4"),
    (6, "Islamabad", 600.0, "C2"),
]

df = spark.createDataFrame(
    data,
    ["id", "city", "amount", "customer_id"]
)

df.createOrReplaceTempView("orders")

spark.sql("""
SELECT
    approx_top_k(city, 3) AS top_cities,
    kll_sketch_get_quantile_double(kll_sketch_agg_double(amount), 0.5) AS median_amount,
    theta_sketch_estimate(theta_sketch_agg(customer_id)) AS distinct_customers_est,
    COUNT(DISTINCT customer_id) AS distinct_customers_exact
FROM orders
""").show(truncate=False)

+-------------------------------------------+-------------+----------------------+------------------------+
|top_cities                                 |median_amount|distinct_customers_est|distinct_customers_exact|
+-------------------------------------------+-------------+----------------------+------------------------+
|[{Lahore, 3}, {Karachi, 2}, {Islamabad, 1}]|300.0        |4                     |4                       |
+-------------------------------------------+-------------+----------------------+------------------------+



## 4.2: `QUALIFY`, pipe aggregates, `time_bucket`, `max_by(…, k)`, `IGNORE NULLS`, vectors, geospatial, CDC

* **`QUALIFY`** filters on window functions without a sub-query ("latest order per customer"). It is a clause of `SELECT`, not a pipe operator (`|> QUALIFY` is a syntax error), but pipe operators can follow it.
* **`time_bucket`**: fixed-size time buckets for dashboards. **`max_by(x, y, k)`** returns the top-k values. **`collect_list(x) IGNORE NULLS`**.
* **Vector functions**: `vector_cosine_similarity`, `vector_l2_distance`, `vector_inner_product`, `vector_norm`, `vector_normalize`, `vector_avg`, `vector_sum`. Simple embedding similarity in SQL.
* **Geospatial GA**: `GEOMETRY(srid)` / `GEOGRAPHY(srid)` types with WKB functions (`st_geomfromwkb`, `st_geogfromwkb`, `st_asbinary`, `st_srid`, `st_setsrid`).
* **`CHANGES`** clause / `DataFrameReader.changes()` for Change Data Capture on catalogs that support it.

In [13]:
from pyspark.sql import functions as F

data = [
    (1, "Lahore",    "C1", 100.0, "2026-10-22 10:01:00", 1.0, 2.0, 31.5204, 74.3587),
    (2, "Lahore",    "C1", 200.0, "2026-10-22 10:03:00", None, 3.0, 31.5204, 74.3587),
    (3, "Karachi",   "C2", 300.0, "2026-10-22 10:06:00", 4.0, None, 24.8607, 67.0011),
    (4, "Karachi",   "C2", 150.0, "2026-10-22 10:08:00", 5.0, 6.0, 24.8607, 67.0011),
    (5, "Islamabad", "C3", 400.0, "2026-10-22 10:11:00", 7.0, 8.0, 33.6844, 73.0479),
    (6, "Islamabad", "C3", 250.0, "2026-10-22 10:13:00", 9.0, 10.0, 33.6844, 73.0479),
]

columns = [
    "id", "city", "customer_id", "amount", "event_time",
    "x", "y", "latitude", "longitude"
]

events = spark.createDataFrame(data, columns)

events = events.withColumn(
    "event_time", F.to_timestamp("event_time")
)

events.createOrReplaceTempView("events")

In [15]:
# QUALIFY — filter after a window function
spark.sql("""
SELECT city, id, amount,
       ROW_NUMBER() OVER (
           PARTITION BY city ORDER BY amount DESC
       ) AS rn
FROM events
QUALIFY rn = 1
ORDER BY city
""").show()

In [16]:
# Pipe AGGREGATE — group and summarize
spark.sql("""
FROM events
|> AGGREGATE SUM(amount) AS revenue,
              COUNT(*) AS order_count
   GROUP BY city
|> ORDER BY revenue DESC
""").show()

+---------+-------+-----------+
|     city|revenue|order_count|
+---------+-------+-----------+
|Islamabad|  650.0|          2|
|  Karachi|  450.0|          2|
|   Lahore|  300.0|          2|
+---------+-------+-----------+



In [18]:
# time_bucket — group timestamps into time intervals
spark.sql("""
SELECT event_time,
       time_bucket(INTERVAL '5' MINUTES, event_time) AS bucket
FROM events
ORDER BY event_time
""").show(truncate=False)

In [ ]:
# max_by(..., k) — get the top K values
spark.sql("""
SELECT city,
       max_by(amount, amount, 2) AS top_amounts
FROM events
GROUP BY city
""").show(truncate=False)

In [ ]:
#IGNORE NULLS — skip missing values
spark.sql("""
SELECT id, city, x,
       LAG(x) IGNORE NULLS OVER (
           PARTITION BY city ORDER BY id
       ) AS previous_x
FROM events
ORDER BY city, id
""").show()

In [ ]:
# Vector functions — compare numeric vectors
spark.sql("""
SELECT
    vector_cosine_similarity(
        array(1.0F, 2.0F),
        array(2.0F, 4.0F)
    ) AS similarity,
    vector_inner_product(
        array(1.0F, 2.0F),
        array(2.0F, 4.0F)
    ) AS dot_product
""").show()

In [ ]:
# Geospatial — create a geographic point
spark.sql("""
SELECT
    city,
    ST_AsText(
        ST_GeomFromText(
            CONCAT('POINT (', longitude, ' ', latitude, ')')
        )
    ) AS location
FROM events
""").show(truncate=False)

In [ ]:
# CDC — read changes between table versions
changes = (
    spark.read
    .option("startingVersion", "1")
    .option("endingVersion", "5")
    .changes("catalog_name.schema_name.orders")
)

changes.show(truncate=False)

## 5. Python UDFs: **Arrow by default** `4.2` · `@arrow_udf` `4.1` · UDF profiler `4.0` benchmark

**What changed**
* **4.0** adds a **unified UDF profiler**: `spark.sql.pyspark.udf.profiler = perf | memory`, then `spark.profile.show()`. It shows where the time goes *inside* your Python UDF on the executors, and it **works over Connect**.
* **4.1** adds **`@arrow_udf`**: the function gets and returns **`pyarrow.Array`s** directly, with no pandas conversion in between.
* **4.2 flips the default:** `spark.sql.execution.pythonUDF.arrow.enabled = true`. A plain **`@udf` now moves data in Arrow batches** (on 3.5/4.0 it pickled row by row). Use `useArrow=False` to get the old behaviour back.

In [21]:
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.functions import udf, pandas_udf

data = [(1, 100.0), (2, 250.0), (3, 500.0), (4, 750.0)]
orders = spark.createDataFrame(data, ["order_id", "amount"])

orders.show()

+--------+------+
|order_id|amount|
+--------+------+
|       1| 100.0|
|       2| 250.0|
|       3| 500.0|
|       4| 750.0|
+--------+------+



In [24]:
# 1. Regular Python UDF: explicitly disable Arrow
@udf("double", useArrow=False)
def tax_regular(amount):
    return round(amount * 1.17, 2)


# 2. Regular Python UDF: uses the configured default
@udf("double")
def tax_default(amount):
    return round(amount * 1.17, 2)


# 3. Pandas UDF: operates on a batch of values
import pandas as pd

@pandas_udf("double")
def tax_pandas(amount: pd.Series) -> pd.Series:
    return (amount * 1.17).round(2)

In [23]:
orders.select(
    "order_id",
    "amount",
    tax_regular("amount").alias("regular_udf"),
    tax_default("amount").alias("default_udf"),
    tax_pandas("amount").alias("pandas_udf")
).show()

+--------+------+-----------+-----------+----------+
|order_id|amount|regular_udf|default_udf|pandas_udf|
+--------+------+-----------+-----------+----------+
|       1| 100.0|      117.0|      117.0|     117.0|
|       2| 250.0|      292.5|      292.5|     292.5|
|       3| 500.0|      585.0|      585.0|     585.0|
|       4| 750.0|      877.5|      877.5|     877.5|
+--------+------+-----------+-----------+----------+



In [26]:
# Arrow's default setting
print(
    spark.conf.get(
        "spark.sql.execution.pythonUDF.arrow.enabled",
        "not explicitly set"
    )
)

print("Regular UDF:", tax_regular.evalType)
print("Default UDF:", tax_default.evalType)
print("Pandas UDF:", tax_pandas.evalType)

In [ ]:
# UDF profiler separately
spark.conf.set("spark.sql.pyspark.udf.profiler", "perf")

@udf("double")
def score(amount):
    return amount * amount

orders.select(score("amount")).show()

spark.profile.show(type="perf")

In [ ]:
# @arrow_udf
import pyarrow as pa
import pyarrow.compute as pc
from pyspark.sql.functions import arrow_udf

@arrow_udf("double")
def tax_arrow(amount: pa.Array) -> pa.Array:
    return pc.round(pc.multiply(amount, 1.17), 2)

orders.select(
    "amount",
    tax_arrow("amount").alias("tax")
).show()

## Python UDTFs (polymorphic) and Arrow-optimized UDFs. [4.0]

In [ ]:
# simple udf
from pyspark.sql.functions import udf

@udf(returnType='int')
def slen(s: str):
    return len(s)

# Arrow udfs
import pyarrow as pa
from pyspark.sql.functions import arrow_udf

@arrow_udf("string")
def to_upper(s: pa.Array) -> pa.Array:
    return pa.compute.ascii_upper(s)

data = [("Alice",), ("Bob",), ("Charlie",)]
df = spark.createDataFrame(data, ["name"])
df.withColumn("name_length", slen(df["name"])).show()

df = spark.createDataFrame([("John Doe",)], ("name",))
df.select(to_upper("name")).show()

In [ ]:
# previous approach in spark 3.5 for udtf 
from pyspark.sql.functions import udtf

@udtf(returnType="value: string, python_type: string")
class FixedInspector:
    def eval(self, value):
        yield str(value), type(value).__name__

spark.udtf.register("fixed_inspector", FixedInspector)

spark.sql("""
    SELECT * FROM fixed_inspector(42)
""").show()

spark.sql("""
    SELECT * FROM fixed_inspector('hello')
""").show()


In [ ]:
from pyspark.sql.functions import udtf
from pyspark.sql.types import StructType, StringType
from pyspark.sql.udtf import AnalyzeArgument, AnalyzeResult

@udtf
class PolymorphicInspector:
    @staticmethod
    def analyze(value: AnalyzeArgument) -> AnalyzeResult:
        # Determine the output type during query analysis
        schema = StructType()
        schema.add("value", value.dataType)
        schema.add("python_type", StringType())

        return AnalyzeResult(schema=schema)

    def eval(self, value):
        # Return the original value without converting it to string
        yield value, type(value).__name__


spark.udtf.register("poly_inspector", PolymorphicInspector)

In [ ]:
# Test 1: Integer input
int_df = spark.sql("""
    SELECT * FROM poly_inspector(42)
""")

int_df.printSchema()
int_df.show()

# Test 2: String input
str_df = spark.sql("""
    SELECT * FROM poly_inspector('hello')
""")

str_df.printSchema()
str_df.show()

# Test 3: Double input
double_df = spark.sql("""
    SELECT * FROM poly_inspector(CAST(3.14 AS DOUBLE))
""")

double_df.printSchema()
double_df.show()

## Python Data Source API: write custom readers and writers in pure Python. [4.0]

In [3]:
from pyspark.sql.datasource import DataSource, DataSourceReader
from pyspark.sql.types import (
    StructType, StructField, IntegerType, StringType
)

class PostsAPIDataSource(DataSource):
    """Read posts from a REST API as a Spark DataFrame."""

    @classmethod
    def name(cls):
        return "posts_api"

    def schema(self):
        return StructType([
            StructField("id", IntegerType(), True),
            StructField("user_id", IntegerType(), True),
            StructField("title", StringType(), True),
            StructField("body", StringType(), True),
        ])

    def reader(self, schema):
        return PostsAPIReader(self.options)


class PostsAPIReader(DataSourceReader):

    def __init__(self, options):
        self.options = options

    def read(self, partition):
        import requests

        url = self.options.get(
            "url",
            "https://jsonplaceholder.typicode.com/posts"
        )
        timeout = int(self.options.get("timeout", "30"))

        response = requests.get(url, timeout=timeout)
        response.raise_for_status()

        records = response.json()

        if not isinstance(records, list):
            raise ValueError("Expected the API to return a JSON array")

        for record in records:
            yield (
                record.get("id"),
                record.get("userId"),
                record.get("title"),
                record.get("body"),
            )

In [4]:
# Register our custom connector
spark.dataSource.register(PostsAPIDataSource)

# Read the API through Spark's standard DataFrameReader
posts_df = (
    spark.read
    .format("posts_api")
    .load()
)

posts_df.printSchema()
posts_df.show(5, truncate=50)

print("Total records:", posts_df.count())

root
 |-- id: integer (nullable = true)
 |-- user_id: integer (nullable = true)
 |-- title: string (nullable = true)
 |-- body: string (nullable = true)



+---+-------+--------------------------------------------------+--------------------------------------------------+
| id|user_id|                                             title|                                              body|
+---+-------+--------------------------------------------------+--------------------------------------------------+
|  1|      1|sunt aut facere repellat provident occaecati ex...|quia et suscipit\nsuscipit recusandae consequun...|
|  2|      1|                                      qui est esse|est rerum tempore vitae\nsequi sint nihil repre...|
|  3|      1|ea molestias quasi exercitationem repellat qui ...|et iusto sed quo iure\nvoluptatem occaecati omn...|
|  4|      1|                              eum et est occaecati|ullam et saepe reiciendis voluptatem adipisci\n...|
|  5|      1|                                nesciunt quas odio|repudiandae veniam quaerat sunt sed\nalias aut ...|
+---+-------+--------------------------------------------------+--------

Total records: 100


## 8. Streaming: state store reader `4.0` · `transformWithState` in Python `4.1` · real-time mode `4.1/4.2`

* **4.0**: the **State Data Source**. `state-metadata` shows operators and batches, `statestore` shows current keys and values, and `readChangeFeed` shows what changed per batch (needs RocksDB + changelog checkpointing).
* **4.0/4.1**: **`transformWithState`**, the next-generation stateful API (multiple named state variables, timers, TTL). **4.1** brings a **row-based Python** version (`groupBy(...).transformWithState(...)`).
* **Real-Time Mode** (4.1 Scala, **4.2 PySpark** `trigger(realTime=...)`): millisecond-latency stateless pipelines, **Kafka source only**, and only allow-listed sinks.

## Stateful Streaming + State Store Reader [4.0]

In [5]:
checkpoint_location = "s3a://buck1/checkpoints/state_demo"

# RocksDB + changelog checkpointing is needed to read the state change feed
spark.conf.set("spark.sql.streaming.stateStore.providerClass",
               "org.apache.spark.sql.execution.streaming.state.RocksDBStateStoreProvider")
spark.conf.set("spark.sql.streaming.stateStore.rocksdb.changelogCheckpointing.enabled", "true")
spark.conf.set("spark.sql.shuffle.partitions", "2")

events = (
    spark.readStream.format("kafka")
    .option("kafka.bootstrap.servers", "broker:9092")
    .option("subscribe", "events")
    .option("startingOffsets", "earliest")
    .load()
    .select(F.from_json(F.col("value").cast("string"),"user STRING, amount INT, ts TIMESTAMP").alias("e"))
    .select("e.*")
)

totals = events.groupBy("user").agg(F.count("*").alias("cnt"), F.sum("amount").alias("total"))

query = (
    totals.writeStream
    .format("memory").queryName("user_totals")
    .outputMode("complete")
    .option("checkpointLocation", checkpoint_location)
    .trigger(processingTime="5 seconds")
    .start()
)

26/10/15 10:50:29 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.
26/10/15 10:50:30 WARN MicroBatchExecution: Disabling AQE since AQE is not supported in stateful workloads.
26/10/15 10:50:51 WARN ProcessingTimeExecutor: Current batch is falling behind. The trigger interval is 5000 milliseconds, but spent 21195 milliseconds


In [10]:
spark.sql("SELECT * FROM user_totals ORDER BY user").show()

+-----+---+-----+
| user|cnt|total|
+-----+---+-----+
|alice| 12|  610|
|  bob|  9|  492|
|carol|  4|  205|
+-----+---+-----+



In [11]:
# State metadata: operators, partitions, batch ids
spark.read.format("state-metadata").load(checkpoint_location).show(truncate=False)

+----------+--------------+--------------+-------------+----------+----------+------------------+
|operatorId|operatorName  |stateStoreName|numPartitions|minBatchId|maxBatchId|operatorProperties|
+----------+--------------+--------------+-------------+----------+----------+------------------+
|0         |stateStoreSave|default       |2            |0         |3         |NULL              |
+----------+--------------+--------------+-------------+----------+----------+------------------+



In [12]:
# Current state (key -> value) of the aggregation, read straight from the checkpoint
spark.read.format("statestore").load(checkpoint_location).show(truncate=False)

+-------+---------+------------+
|key    |value    |partition_id|
+-------+---------+------------+
|{bob}  |{9, 492} |1           |
|{alice}|{12, 610}|1           |
|{carol}|{4, 205} |1           |
+-------+---------+------------+



In [14]:
# Change feed of the state between batches (PUT / DELETE per key)
(spark.read.format("statestore")
    .option("readChangeFeed", True)
    .option("changeStartBatchId", 0)
    .load(checkpoint_location)
    .orderBy("batch_id")
    .show(truncate=False))

+--------+-----------+-------+---------+------------+
|batch_id|change_type|key    |value    |partition_id|
+--------+-----------+-------+---------+------------+
|0       |update     |{bob}  |{5, 279} |1           |
|0       |update     |{alice}|{6, 274} |1           |
|0       |update     |{carol}|{4, 205} |1           |
|1       |update     |{alice}|{8, 412} |1           |
|2       |update     |{bob}  |{8, 408} |1           |
|2       |update     |{alice}|{10, 556}|1           |
|3       |update     |{alice}|{12, 610}|1           |
|3       |update     |{bob}  |{9, 492} |1           |
+--------+-----------+-------+---------+------------+



In [ ]:
query.stop()

In [ ]:
# 4.1: transformWithState in Python (row-based). Named state variable + per-key logic; then read that state back by name.
from pyspark.sql import Row
from pyspark.sql.streaming.stateful_processor import StatefulProcessor, StatefulProcessorHandle

class SeenCounter(StatefulProcessor):
    def init(self, handle: StatefulProcessorHandle):
        self.seen = handle.getValueState("seen", StructType([StructField("n", LongType())]))
    def handleInputRows(self, key, rows, timerValues):
        n = (self.seen.get()[0] if self.seen.exists() else 0) + sum(1 for _ in rows)
        self.seen.update((n,))
        yield Row(user=key[0], seen=n)
    def close(self): pass

TWS = "/data/checkpoints/tws"
q2 = (events.groupBy("user")
      .transformWithState(SeenCounter(), outputStructType="user STRING, seen LONG", outputMode="Update", timeMode="None")
      .writeStream.format("memory").queryName("tws_out").outputMode("update").option("checkpointLocation", TWS).start())
t0 = time.time()
while time.time() - t0 < 60 and not (q2.lastProgress and q2.lastProgress["batchId"] >= 3):   # first batch starts Python workers: wait for 3+ commits
    time.sleep(2)
q2.stop()
spark.sql("SELECT user, max(seen) AS seen FROM tws_out GROUP BY user ORDER BY user").show()
spark.read.format("statestore").option("stateVarName", "seen").load(TWS).show(truncate=False)
check("Streaming", "transformWithState (Python, row-based)", lambda: spark.sql("SELECT count(*) FROM tws_out").collect(), since="4.1");

In [ ]:

from pyspark.sql import SparkSession
from pyspark.sql.streaming.stateful_processor import (
    StatefulProcessor,
    StatefulProcessorHandle,
)
from pyspark.sql.types import (
    StructType, StructField, StringType, LongType
)

spark = SparkSession.builder.getOrCreate()

output_schema = StructType([
    StructField("user_id", StringType()),
    StructField("total_events", LongType())
])

class UserEventCounter(StatefulProcessor):

    def init(self, handle: StatefulProcessorHandle):
        self.count = handle.getValueState(
            "event_count",
            StructType([
                StructField("n", LongType())
            ])
        )

    def handleInputRows(self, key, rows, timerValues):
        # Retrieve the previous count
        total = self.count.get()[0] if self.count.exists() else 0

        # Count all events for this user in this batch
        batch_count = sum(1 for _ in rows)

        # Update persistent state
        total += batch_count
        self.count.update((total,))

        # Emit the updated count
        yield (key[0], total)

    def close(self):
        pass

# events is a streaming DataFrame with:
# user_id, event

result = (
    events.groupBy("user_id")
    .transformWithState(
        statefulProcessor=UserEventCounter(),
        outputStructType=output_schema,
        outputMode="Update",
        timeMode="None"
    )
)

query = (
    result.writeStream
    .outputMode("update")
    .option("checkpointLocation", "/tmp/user-event-checkpoint")
    .format("console")
    .start()
)

query.awaitTermination()

In [ ]:
# 4.2: the real-time trigger is in the PySpark API, but needs a Kafka source + allow-listed sink. The rate source is rejected.
def try_real_time():
    q = events.writeStream.format("console").outputMode("update").trigger(realTime="5 seconds").start()
    q.awaitTermination(8)
    err = q.exception(); q.stop()
    if err: raise err
check("Streaming", "trigger(realTime=...) with the rate source", try_real_time, since="4.2", note="real-time mode supports Kafka sources only");

## Continuous vs Realtime mode

In [7]:
checkpoint_location = "s3a://buck1/checkpoints/continuous_demo_v677"

events = (
    spark.readStream.format("kafka")
    .option("kafka.bootstrap.servers", "broker:9092")
    # topic has 3 partitions but this session only has 2 cores; both modes need 1 core per partition
    .option("assign", '{"events":[0]}')
    .option("startingOffsets", "earliest")
    .load()
    .select(F.from_json(F.col("value").cast("string"),
                        "user STRING, amount INT, ts TIMESTAMP").alias("e"))
    .select("e.*")
)

# Continuous mode only supports map-like ops (select/filter/from_json), no aggregations
filtered = events.filter(F.col("amount") > 50)

query = (
    filtered.writeStream
    .format("memory").queryName("continuous_events")
    .outputMode("append")
    .option("checkpointLocation", checkpoint_location)
    .trigger(continuous="5 seconds")
    .start()
)

26/10/21 05:58:46 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.
26/10/21 05:58:46 WARN ContinuousExecution: Disabling AQE since the query runs with continuous mode.


**Continuous:** expect **one job and one stage** for the whole run. Its tasks stay `running` and never finish, and no new jobs appear. The plan shows `ContinuousScan` → `WriteToContinuousDataSource`, and `lastProgress` is `None` because there are no batches.

In [13]:
spark.sql("SELECT * FROM continuous_events").show(truncate=False)

+-----+------+--------------------------+
|user |amount|ts                        |
+-----+------+--------------------------+
|alice|76    |2026-10-21 05:56:43.341825|
|bob  |98    |2026-10-21 05:56:46.34656 |
|alice|61    |2026-10-21 05:56:54.378352|
|bob  |67    |2026-10-21 05:56:55.382728|
|bob  |55    |2026-10-21 05:59:18.4168  |
|alice|61    |2026-10-21 05:59:27.454417|
|alice|90    |2026-10-21 05:59:34.476106|
|alice|62    |2026-10-21 05:59:36.481824|
|alice|69    |2026-10-21 05:59:41.497938|
+-----+------+--------------------------+



In [15]:
# Watch the jobs/stages this query creates (also visible live at http://localhost:4040 -> Jobs / Stages)
import time

def watch_stages(query, seconds=30):
    st = spark.sparkContext.statusTracker()
    t0 = time.time()
    while time.time() - t0 < seconds:
        jobs = sorted(st.getJobIdsForGroup(str(query.runId)))
        stages = []
        for sid in st.getActiveStageIds():
            s = st.getStageInfo(sid)
            if s:
                stages.append(f"stage {sid}: {s.numActiveTasks} running, {s.numCompletedTasks}/{s.numTasks} done")
        print(f"t={time.time() - t0:4.0f}s  jobs so far={jobs}  " + " | ".join(stages))
        time.sleep(1)

watch_stages(query)

t=   0s  jobs so far=[3]  
t=   1s  jobs so far=[3]  
t=   2s  jobs so far=[3]  
t=   3s  jobs so far=[3]  
t=   4s  jobs so far=[3]  
t=   5s  jobs so far=[3]  
t=   6s  jobs so far=[3]  
t=   7s  jobs so far=[3]  
t=   8s  jobs so far=[3]  
t=   9s  jobs so far=[3]  
t=  10s  jobs so far=[3]  
t=  11s  jobs so far=[3]  
t=  12s  jobs so far=[3]  
t=  13s  jobs so far=[3]  
t=  14s  jobs so far=[3]  
t=  15s  jobs so far=[3]  
t=  16s  jobs so far=[3]  
t=  17s  jobs so far=[3]  
t=  18s  jobs so far=[3]  
t=  19s  jobs so far=[3]  
t=  20s  jobs so far=[3]  
t=  21s  jobs so far=[3]  
t=  22s  jobs so far=[3]  
t=  23s  jobs so far=[3]  
t=  24s  jobs so far=[3]  
t=  25s  jobs so far=[3]  
t=  26s  jobs so far=[3]  
t=  27s  jobs so far=[3]  
t=  28s  jobs so far=[3]  
t=  29s  jobs so far=[3]  


In [16]:
query.explain()
print("last progress:", query.lastProgress and {k: query.lastProgress[k] for k in ("batchId", "numInputRows")})

== Physical Plan ==
WriteToContinuousDataSource org.apache.spark.sql.execution.streaming.sources.MemoryStreamingWrite@3c6e1683
+- *(1) Project [e#51.user AS user#52, e#51.amount AS amount#53, e#51.ts AS ts#54]
   +- Project [from_json(StructField(user,StringType,true), StructField(amount,IntegerType,true), StructField(ts,TimestampType,true), cast(value#45 as string), Some(Etc/UTC), false) AS e#51]
      +- Filter (isnotnull(value#45) AND (from_json(StructField(amount,IntegerType,true), cast(value#45 as string), Some(Etc/UTC), false).amount > 50))
         +- ContinuousScan[key#44, value#45, topic#46, partition#47, offset#48L, timestamp#49, timestampType#50] class org.apache.spark.sql.kafka010.KafkaSourceProvider$KafkaScan


last progress: None


In [ ]:
query.stop()

In [22]:
checkpoint_location = "s3a://buck1/checkpoints/realtime_demo_v33"

events = (
    spark.readStream.format("kafka")
    .option("kafka.bootstrap.servers", "broker:9092")
    # topic has 3 partitions but this session only has 2 cores; both modes need 1 core per partition
    .option("assign", '{"events":[0,1]}')
    .option("startingOffsets", "earliest")
    .load()
    .select(F.from_json(F.col("value").cast("string"),
                        "user STRING, amount INT, ts TIMESTAMP").alias("e"))
    .select("e.*")
)

# Real-time mode (4.1) is stateless only: project/filter/union/broadcast join, no aggregations
writer = (
    events.filter(F.col("amount") > 50)
    .writeStream
    .format("console")
    .outputMode("update")
    .option("checkpointLocation", checkpoint_location)
)

# PySpark 4.1.0 doesn't expose trigger(realTime=...), so set the JVM trigger directly
jvm = spark.sparkContext._jvm
writer._jwrite = writer._jwrite.trigger(jvm.org.apache.spark.sql.streaming.Trigger.RealTime("1 minutes"))
query = writer.start()

26/10/21 06:25:19 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.
26/10/21 06:25:19 WARN MicroBatchExecution: Disabling AQE since AQE is not supported for Real-time Mode.


-------------------------------------------
Batch: 0
-------------------------------------------
+-----+------+--------------------+
| user|amount|                  ts|
+-----+------+--------------------+
|alice|    76|2026-10-21 05:56:...|
|  bob|    98|2026-10-21 05:56:...|
|alice|    61|2026-10-21 05:56:...|
|  bob|    67|2026-10-21 05:56:...|
|  bob|    55|2026-10-21 05:59:...|
|alice|    61|2026-10-21 05:59:...|
|alice|    90|2026-10-21 05:59:...|
|alice|    62|2026-10-21 05:59:...|
|alice|    69|2026-10-21 05:59:...|
|carol|    80|2026-10-21 05:59:...|
|carol|    96|2026-10-21 06:00:...|
|  bob|    93|2026-10-21 06:00:...|
|carol|    99|2026-10-21 06:00:...|
|carol|    80|2026-10-21 06:00:...|
|  bob|    60|2026-10-21 06:00:...|
|  bob|    93|2026-10-21 06:00:...|
|carol|    56|2026-10-21 06:00:...|
|  bob|    60|2026-10-21 06:00:...|
|carol|    80|2026-10-21 06:00:...|
|alice|    64|2026-10-21 06:01:...|
+-----+------+--------------------+
only showing top 20 rows


-------------------------------------------
Batch: 1
-------------------------------------------
+-----+------+--------------------+
| user|amount|                  ts|
+-----+------+--------------------+
|  bob|    94|2026-10-21 06:26:...|
|  bob|    63|2026-10-21 06:26:...|
|carol|    96|2026-10-21 06:26:...|
|carol|    59|2026-10-21 06:26:...|
|  bob|    94|2026-10-21 06:26:...|
|carol|    58|2026-10-21 06:26:...|
|alice|    87|2026-10-21 06:27:...|
|alice|    80|2026-10-21 06:27:...|
|alice|    65|2026-10-21 06:27:...|
|carol|    90|2026-10-21 06:27:...|
|carol|    74|2026-10-21 06:27:...|
|alice|    56|2026-10-21 06:27:...|
|carol|    60|2026-10-21 06:26:...|
|  bob|    97|2026-10-21 06:26:...|
|  bob|    71|2026-10-21 06:26:...|
|alice|    95|2026-10-21 06:26:...|
|carol|    94|2026-10-21 06:26:...|
|carol|    87|2026-10-21 06:26:...|
|carol|    64|2026-10-21 06:26:...|
|  bob|    66|2026-10-21 06:26:...|
+-----+------+--------------------+
only showing top 20 rows


-------------------------------------------
Batch: 2
-------------------------------------------
+-----+------+--------------------+
| user|amount|                  ts|
+-----+------+--------------------+
|alice|    69|2026-10-21 06:27:...|
|alice|    69|2026-10-21 06:27:...|
|alice|    64|2026-10-21 06:27:...|
|alice|    71|2026-10-21 06:27:...|
|  bob|    93|2026-10-21 06:27:...|
|  bob|    78|2026-10-21 06:27:...|
|alice|    94|2026-10-21 06:27:...|
|  bob|    72|2026-10-21 06:27:...|
|alice|   100|2026-10-21 06:27:...|
|  bob|    98|2026-10-21 06:27:...|
|  bob|    61|2026-10-21 06:27:...|
|carol|    84|2026-10-21 06:28:...|
+-----+------+--------------------+



-------------------------------------------
Batch: 3
-------------------------------------------
+----+------+---+
|user|amount| ts|
+----+------+---+
+----+------+---+



...

-------------------------------------------
Batch: 8
-------------------------------------------
+----+------+---+
|user|amount| ts|
+----+------+---+
+----+------+---+



-------------------------------------------
Batch: 9
-------------------------------------------
+----+------+---+
|user|amount| ts|
+----+------+---+
+----+------+---+



-------------------------------------------
Batch: 10
-------------------------------------------
+----+------+---+
|user|amount| ts|
+----+------+---+
+----+------+---+



-------------------------------------------
Batch: 11
-------------------------------------------
+----+------+---+
|user|amount| ts|
+----+------+---+
+----+------+---+



-------------------------------------------
Batch: 12
-------------------------------------------
+----+------+---+
|user|amount| ts|
+----+------+---+
+----+------+---+



**Real-time:** expect a **new job about every 5 s** (`jobs so far` grows: [0], [0, 1], [0, 1, 2]...). Each job's tasks run for the 5 s batch, finish, and the next job starts. The plan shows `RealTimeStreamScan [batchDurationMs=5000ms]` → `WriteToDataSourceV2` (the normal micro-batch writer), and `lastProgress` has a `batchId`.

In [ ]:
watch_stages(query)

In [ ]:
query.explain()
print("last progress:", query.lastProgress and {k: query.lastProgress[k] for k in ("batchId", "numInputRows")})

In [20]:
query.stop()

26/10/21 06:25:07 ERROR WriteToDataSourceV2Exec: Data source write support MicroBatchWrite[epoch: 0, writer: ConsoleWriter[numRows=20, truncate=true]] is aborting.
26/10/21 06:25:07 ERROR WriteToDataSourceV2Exec: Data source write support MicroBatchWrite[epoch: 0, writer: ConsoleWriter[numRows=20, truncate=true]] aborted.
26/10/21 06:25:07 WARN DAGScheduler: Failed to cancel job group 4fa84f0d-7d8c-472a-940f-57363bb3c2a8. Cannot find active jobs for it.
26/10/21 06:25:07 WARN TaskSetManager: Lost task 1.0 in stage 31.0 (TID 48) (172.18.0.11 executor 0): TaskKilled (Stage cancelled: [SPARK_JOB_CANCELLED] Job 31 cancelled Query [id = 2772029f-cd7f-48bd-9de5-3565b0c9730c, runId = 4fa84f0d-7d8c-472a-940f-57363bb3c2a8] was stopped SQLSTATE: XXKDA)
26/10/21 06:25:07 WARN TaskSetManager: Lost task 0.0 in stage 31.0 (TID 47) (172.18.0.11 executor 0): TaskKilled (Stage cancelled: [SPARK_JOB_CANCELLED] Job 31 cancelled Query [id = 2772029f-cd7f-48bd-9de5-3565b0c9730c, runId = 4fa84f0d-7d8c-472a-

## Plain Text Logging vs Structured (JSON) Logging [4.0]

In [27]:
import subprocess

with open("/tmp/log_job.py", "w") as f:
    f.write(
        "from pyspark.sql import SparkSession\n"
        "s = SparkSession.builder.master('local[1]').getOrCreate()\n"
        "s.range(5).count()\n"
        "s.stop()\n"
    )

def run_job(structured):
    r = subprocess.run(
        ["spark-submit", "--conf", f"spark.log.structuredLogging.enabled={structured}", "/tmp/log_job.py"],
        capture_output=True, text=True,
    )
    return [l for l in r.stderr.splitlines() if "SparkContext" in l]

plain_logs = run_job("false")
json_logs = run_job("true")

In [28]:
# Plain text: the values are buried inside the message
print("\n".join(plain_logs[:3]))

26/10/22 06:51:26 INFO SparkContext: Running Spark version 4.1.0
26/10/22 06:51:26 INFO SparkContext: OS info Linux, 6.12.54-linuxkit, amd64
26/10/22 06:51:26 INFO SparkContext: Java version 21.0.9+10-LTS


In [29]:
# Structured: one JSON object per line, values also in the "context" map
print("\n".join(json_logs[:3]))

{"ts":"2026-10-22T06:51:58.585Z","level":"INFO","msg":"Running Spark version 4.1.0","context":{"spark_version":"4.1.0"},"logger":"SparkContext"}
{"ts":"2026-10-22T06:51:58.593Z","level":"INFO","msg":"OS info Linux, 6.12.54-linuxkit, amd64","context":{"os_arch":"amd64","os_name":"Linux","os_version":"6.12.54-linuxkit"},"logger":"SparkContext"}
{"ts":"2026-10-22T06:51:58.594Z","level":"INFO","msg":"Java version 21.0.9+10-LTS","context":{"java_version":"21.0.9+10-LTS"},"logger":"SparkContext"}


In [30]:
# JSON logs can be queried like any table using SPARK_LOG_SCHEMA
from pyspark.logger import SPARK_LOG_SCHEMA

logs_df = (
    spark.createDataFrame([(l,) for l in json_logs if l.startswith("{")], "value STRING")
    .select(F.from_json("value", SPARK_LOG_SCHEMA).alias("log"))
    .select("log.*")
)

logs_df.select("ts", "level", "logger", "msg", "context").show(5, truncate=60)
logs_df.where(F.col("context")["spark_version"].isNotNull()).select("context").show(truncate=False)

+-----------------------+-----+------------+------------------------------------------------------------+------------------------------------------------------------+
|                     ts|level|      logger|                                                         msg|                                                     context|
+-----------------------+-----+------------+------------------------------------------------------------+------------------------------------------------------------+
|2026-10-22 06:51:58.585| INFO|SparkContext|                                 Running Spark version 4.1.0|                                    {spark_version -> 4.1.0}|
|2026-10-22 06:51:58.593| INFO|SparkContext|                      OS info Linux, 6.12.54-linuxkit, amd64|{os_arch -> amd64, os_name -> Linux, os_version -> 6.12.5...|
|2026-10-22 06:51:58.594| INFO|SparkContext|                                  Java version 21.0.9+10-LTS|                             {java_version -> 21.0.9+10-LTS}

## Spark Connect Evolution [4.0-4.2]
* The client talks to a remote Spark server over gRPC, so there is **no JVM in your Python process**. Client crashes don't kill the driver, and many notebooks share one cluster.
* **4.0**: near-full API parity, `spark.api.mode=connect|classic`, slim **`pyspark-client`** package (~1.5 MB).
* **4.1**: **Spark ML on Connect GA**, a Connect **JDBC driver**, compressed plans, chunked Arrow results.
* **4.2**: `DataFrame.toJSON`, `zipWithIndex`, `GetStatus` API, `spark.read.json/csv/xml` on a DataFrame.
* **Still not available over Connect (by design):** `spark._jvm`, `sparkContext` (so `setLogLevel`, broadcast/accumulators), RDDs.

In [ ]:
# New lightweight pyspark-client package (about 1.5 MB).
from pyspark.sql import SparkSession

spark = SparkSession.builder.remote("sc://localhost:15002").getOrCreate()

In [ ]:
from datetime import datetime, date
from pyspark.sql import Row

df = spark.createDataFrame([
    Row(a=1, b=2., c='string1', d=date(2000, 1, 1), e=datetime(2000, 1, 1, 12, 0)),
    Row(a=2, b=3., c='string2', d=date(2000, 2, 1), e=datetime(2000, 1, 2, 12, 0)),
    Row(a=4, b=5., c='string3', d=date(2000, 3, 1), e=datetime(2000, 1, 3, 12, 0))
])
df.show()


In [ ]:
df.count()